In [2]:
#! run the cell below before running the interactive demo
o = InteractiveSegment('./WikiArt/17.jpg')
o.render()

In [1]:
#* run this first

%matplotlib tk
import matplotlib.pyplot as plt
from IPython.display import clear_output
import matplotlib.image as mpimg
from segment_anything import SamAutomaticMaskGenerator, sam_model_registry, SamPredictor
import numpy as np
import torch
from icecream import ic
import ipywidgets as widgets
import os
from utils import *
import PIL
from fast_pytorch_kmeans import KMeans
import glob
class InteractiveSegment():
    def __init__(self,style_img_path):
        self.input_points = []
        self.input_labels = []
        self.final_mask = None

        
        self.style_name = os.path.basename(style_img_path).split('.')[0]
        self.img= mpimg.imread(style_img_path)
        self.fig = plt.figure(figsize=(15,10))
        sam = sam_model_registry["default"](checkpoint="./sam_vit_h_4b8939.pth").to("cuda")
        self.predictor = SamPredictor(sam)
        self.predictor.set_image(self.img)
        
        self.select_all = False
        
        #GUI
        self.button1 = widgets.Button(description="Clear")
        self.button2 = widgets.Button(description="Save")
        self.button3 = widgets.Button(description="Select All")
        
        default_save_dir = f'./styles/{self.style_name}/'
        possible_paths = sorted(glob.glob(default_save_dir+"style_*"))
        if len(possible_paths) > 0:
            ind = int(possible_paths[-1].split('_')[-1])
            default_save_dir = default_save_dir + f'style_{ind+1}'
        else:
            default_save_dir = default_save_dir + 'style_1'

            
        self.textBox = widgets.Text(value=default_save_dir, disabled=False, description='Save Path:')
        self.out = widgets.Output()
        buttons = widgets.VBox(children=[self.button1,self.button2,self.button3,self.textBox])
        self.all_widgets = widgets.HBox(children=[buttons, self.out])
        display(self.all_widgets)

    def save(self,b):
        save_dir = self.textBox.value
        ensure_dirs(save_dir)
        if not self.select_all:
            self.final_mask = self.final_mask.reshape(self.img.shape[0], self.img.shape[1])
            top_left, bottom_right = find_min_bounding_box(self.final_mask)
            valid_mask = self.final_mask[top_left[0]:bottom_right[0], top_left[1]:bottom_right[1]]
            valid_img = self.img[top_left[0]:bottom_right[0], top_left[1]:bottom_right[1],:]
            masked_img = valid_img*valid_mask[..., None]
            # masked_img = self.img*self.final_mask[..., None]
            # valid_color_arr = self.img[self.final_mask==1,:]/255.0
            valid_color_arr = valid_img[valid_mask==1,:]/255.0
            avg_color = np.mean(valid_color_arr, axis=0)
            color_patch = np.ones((50,50,3),dtype=np.float32)*avg_color
            PIL.Image.fromarray((color_patch*255).astype(np.uint8)).save(os.path.join(save_dir, f'{self.style_name}_color.png'))
            PIL.Image.fromarray(masked_img).save(os.path.join(save_dir, f'{self.style_name}_masked.png'))
            PIL.Image.fromarray(self.img).save(os.path.join(save_dir, f'img.png'))
            PIL.Image.fromarray(valid_img).save(os.path.join(save_dir, f'valid_img.png'))
            np.save(os.path.join(save_dir, 'valid_mask.npy'), valid_mask)
            np.save(os.path.join(save_dir, 'mask.npy'), self.final_mask)
            np.save(os.path.join(save_dir, 'color.npy'), avg_color)
            print(f"Saved to {save_dir} successfully!")
        else:
            # self.final_mask = self.final_mask.reshape(self.img.shape[0], self.img.shape[1])
            self.final_mask = np.ones((self.img.shape[0], self.img.shape[1]))
            valid_mask = self.final_mask
            valid_img = self.img
            masked_img = valid_img#*valid_mask[..., None]
            valid_color_arr = valid_img[valid_mask==1,:]/255.0
            avg_color = np.mean(valid_color_arr, axis=0)
            color_patch = np.ones((50,50,3),dtype=np.float32)*avg_color
            PIL.Image.fromarray((color_patch*255).astype(np.uint8)).save(os.path.join(save_dir, f'{self.style_name}_color.png'))
            PIL.Image.fromarray(masked_img).save(os.path.join(save_dir, f'{self.style_name}_masked.png'))
            PIL.Image.fromarray(self.img).save(os.path.join(save_dir, f'img.png'))
            PIL.Image.fromarray(valid_img).save(os.path.join(save_dir, f'valid_img.png'))
            np.save(os.path.join(save_dir, 'valid_mask.npy'), valid_mask)
            np.save(os.path.join(save_dir, 'mask.npy'), self.final_mask)
            np.save(os.path.join(save_dir, 'color.npy'), avg_color)
            print(f"Saved to {save_dir} successfully! (Select All)")
        

    def clear(self,b):
        self.input_points = []
        self.input_labels = []
        self.clear_mask(plt.gca())
        clear_output(wait=True)
        display(self.all_widgets)
        plt.axis('off')
        plt.show()
        
    def selectAll(self,b):
        self.select_all = not self.select_all


    def show_mask(self,mask, ax, random_color=False):
        if random_color:
            color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
        else:
            color = np.array([30/255, 144/255, 255/255, 0.6])
        h, w = mask.shape[-2:]
        mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
        ax.imshow(mask_image)
        
    def clear_mask(self,ax):
        ax.cla()
        ax.imshow(self.img)
        for idx in range(len(self.input_points)):
            self.show_points(np.array([self.input_points[idx]]), np.array([self.input_labels[idx]]), ax)
        
        
    def show_points(self,coords, labels, ax, marker_size=375):
        pos_points = coords[labels==1]
        neg_points = coords[labels==0]
        ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
        ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   
        

    def onclick(self,event): #left click add points for adding mask, right click add points for removing mask
        ix, iy = event.xdata, event.ydata
        self.input_points.append([ix, iy])
        if event.button == 1:
            self.input_labels.append(1)
            self.show_points(np.array([[ix, iy]]), np.array([1]), plt.gca())
        elif event.button == 3: # it is right-click
            self.show_points(np.array([[ix, iy]]), np.array([0]), plt.gca())
            self.input_labels.append(0)
        
        masks, scores, logits = self.predictor.predict(
            point_coords=np.array(self.input_points),
            point_labels=np.array(self.input_labels),
            multimask_output=False,
        )
        self.clear_mask(plt.gca())
        print(f"input coordinates: x = {ix:02f}; y = {iy:02f}")
        self.final_mask = masks
        self.show_mask(masks, plt.gca())
        plt.axis('off')
        plt.show()
        
    def render(self):
        self.button1.on_click(self.clear)
        self.button2.on_click(self.save)
        self.button3.on_click(self.selectAll)
        with self.out:
            cid = self.fig.canvas.mpl_connect('button_press_event', self.onclick)
            imgplot = plt.imshow(self.img)
            plt.axis('off')
            plt.show()
            
def find_min_bounding_box(mask):
    mask = mask.astype(bool)
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return (rmin, cmin), (rmax, cmax)